In [54]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [55]:
# MobileNet
mob_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
mob_out = GlobalAveragePooling2D()(mob_base.output)
mob_model = Model(inputs=mob_base.input, outputs=mob_out)

# EfficientNet
eff_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
eff_out = GlobalAveragePooling2D()(eff_base.output)
eff_model = Model(inputs=eff_base.input, outputs=eff_out)

In [56]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)

    # Resize
    img = cv2.resize(img, (224,224))

    # Convert to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Prepare for both models
    img_mob = mob_pre(img.copy())
    img_eff = eff_pre(img.copy())

    return np.expand_dims(img_mob, axis=0), np.expand_dims(img_eff, axis=0)

In [57]:
def extract_features(img_path):
    img_mob, img_eff = preprocess_image(img_path)

    f_mob = mob_model.predict(img_mob, verbose=0)
    f_eff = eff_model.predict(img_eff, verbose=0)

    return f_mob.flatten(), f_eff.flatten()

In [58]:
def fuse_features(f_mob, f_eff):
    return np.concatenate([f_mob, f_eff])   # final vector

In [59]:
import os

X = []
y = []

dataset_path = "outputs/"   # structure: dataset/normal, dataset/tumor

for label, folder in enumerate(["no", "yes"]):
    folder_path = os.path.join(dataset_path, folder)

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)

        f_mob, f_eff = extract_features(img_path)
        fused = fuse_features(f_mob, f_eff)

        X.append(fused)
        y.append(label)

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

(253, 2560) (253,)


In [60]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.7450980392156863


In [52]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))

[[15  9]
 [ 3 24]]


In [53]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.62      0.71        24
           1       0.73      0.89      0.80        27

    accuracy                           0.76        51
   macro avg       0.78      0.76      0.76        51
weighted avg       0.78      0.76      0.76        51



In [22]:
import numpy as np

# Example: 2560 features → 40 timesteps × 64 features
timesteps = 40
features_per_step = X.shape[1] // timesteps

X_lstm = X[:, :timesteps * features_per_step]  # trim if needed
X_lstm = X_lstm.reshape(X.shape[0], timesteps, features_per_step)

print(X_lstm.shape)

(253, 40, 64)


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_lstm, y, test_size=0.2)

In [24]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True, input_shape=(timesteps, features_per_step)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Users\chara\Downloads\opti project\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [33]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [34]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=8,
    validation_data=(X_test, y_test)
)

Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.9604 - loss: 0.1155 - val_accuracy: 0.8824 - val_loss: 0.9058
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9851 - loss: 0.0496 - val_accuracy: 0.9216 - val_loss: 0.4337
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9950 - loss: 0.0153 - val_accuracy: 0.9216 - val_loss: 0.4367
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9950 - loss: 0.0065 - val_accuracy: 0.9216 - val_loss: 0.4092
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 1.0000 - loss: 0.0034 - val_accuracy: 0.9216 - val_loss: 0.4756
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.9216 - val_loss: 0.4584
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 1.0000 - loss: 8.0773e-04 - val_accuracy: 0.9216 - val_loss: 0.5307
Epoch 8/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 1.0000 - loss: 4.7759e-04 - val_accuracy: 0.

In [35]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9216 - loss: 0.6078
Test Accuracy: 0.9215686321258545


In [37]:
print(np.bincount(y))

[ 98 155]


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(1, 224, 224, 3), dtype=float32) with name 'keras_tensor_788' and path ''. Expected shape (None, 40, 64), but input has incompatible shape (1, 224, 224, 3)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [40]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base (important for small dataset)
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=base.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [41]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    "dataset/",
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

val = datagen.flow_from_directory(
    "dataset/",
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

Found 203 images belonging to 2 classes.
Found 50 images belonging to 2 classes.


In [42]:
model.fit(
    train,
    validation_data=val,
    epochs=10
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.5468 - loss: 0.6912 - val_accuracy: 0.6200 - val_loss: 0.6812
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 286ms/step - accuracy: 0.5567 - loss: 0.6988 - val_accuracy: 0.6200 - val_loss: 0.6654
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 281ms/step - accuracy: 0.5961 - loss: 0.6837 - val_accuracy: 0.6200 - val_loss: 0.6690
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 272ms/step - accuracy: 0.6059 - loss: 0.6809 - val_accuracy: 0.6200 - val_loss: 0.6634
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 354ms/step - accuracy: 0.6207 - loss: 0.6714 - val_accuracy: 0.6200 - val_loss: 0.6680
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 278ms/step - accuracy: 0.5764 - loss: 0.7003 - val_accuracy: 0.6200 - val_loss: 0.6730
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 277ms/step - accuracy: 0.5961 - loss: 0.6754 - val_accuracy: 0.6200 - val_loss: 0.6667
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 275ms/step - accuracy: 0.6059 - loss: 0.6897 - val_accuracy: 0

In [43]:
model.evaluate(val)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 195ms/step - accuracy: 0.6200 - loss: 0.6647


[0.6647453308105469, 0.6200000047683716]

In [44]:
import cv2
import numpy as np

# Load image
img = cv2.imread("test.jpg")

# Resize
img = cv2.resize(img, (224, 224))

# Normalize
img = img / 255.0
# Expand dimensions (important)
img = np.expand_dims(img, axis=0)

# Predict
pred = model.predict(img)

if pred[0][0] > 0.5:
    print("Tumor")
else:
    print("No Tumor")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Tumor
